# Visual History Agent — Colab (02 → 04)

Downloads **one self-contained public zip** with `gdown` (**no Drive mount**).
The zip includes weights, Formal Evidence atom JSON, sparse codes, scripts, and
vendored `vendor/VisualHistory` (LLM gateway). Runtime paths are project-relative only.

### Author (once)
```bash
bash scripts/pack_colab_zip.sh
# → ../Visual_History_Agent.zip
```
Upload/replace on Drive → **Anyone with the link → Viewer** → file id in config cell.

### Runtime
- Runtime → **GPU**
- `OPENAI_API_KEY` is set in the config cell


In [ ]:
# Google Drive FILE id for Visual_History_Agent.zip (Anyone with the link → Viewer)
# Replace if you host your own zip.
DRIVE_ZIP_FILE_ID = "1wDTHIIUtog4cvDHL_2VJ4rlF-5gaDQPh"

import os
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = (userdata.get("OPENAI_API_KEY") or "").strip()
    except Exception:
        OPENAI_API_KEY = ""
# Private local paste only — do not commit a real key:
# OPENAI_API_KEY = OPENAI_API_KEY or "sk-..."
assert OPENAI_API_KEY, "Set OPENAI_API_KEY as a Colab secret or environment variable"


## 1 — Download zip + install


In [ ]:
import os
import sys
import shutil
import zipfile
from pathlib import Path

IN_COLAB = Path("/content").is_dir()
WORK = Path("/content") if IN_COLAB else Path(".").resolve()
ROOT = WORK / "Visual_History_Agent"
ZIP_PATH = WORK / "Visual_History_Agent.zip"

%pip install -q -U "gdown>=5.2"
import gdown
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| gdown", getattr(gdown, "__version__", "?"))

if not DRIVE_ZIP_FILE_ID or str(DRIVE_ZIP_FILE_ID).startswith("PASTE_"):
    raise ValueError("Set DRIVE_ZIP_FILE_ID to the public Drive zip file id")

if ROOT.exists():
    shutil.rmtree(ROOT)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print("Downloading zip", DRIVE_ZIP_FILE_ID)
out = gdown.download(id=DRIVE_ZIP_FILE_ID, output=str(ZIP_PATH), quiet=False)
if not out or not ZIP_PATH.is_file():
    raise RuntimeError("gdown failed — confirm Anyone with the link → Viewer")

print("Extracting", ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(WORK)

if not ((ROOT / "visual_history_agent").is_dir() and (ROOT / "data").is_dir()):
    raise FileNotFoundError(f"Incomplete extract under {ROOT}")

os.environ["VISUAL_HISTORY_AGENT_ROOT"] = str(ROOT)
os.environ["VISUALHISTORY_ROOT"] = str(ROOT / "vendor")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "vendor"))
os.chdir(ROOT)

%pip install -q -r requirements-colab.txt
%pip install -q -e .

from visual_history_agent.paths import require_downstream_artifacts, visualhistory_root

paths_check = require_downstream_artifacts(ROOT)
vh = visualhistory_root(ROOT)
import faiss
from VisualHistory.llm import get_client  # noqa: F401

for rel in (
    "scripts/infer_single_image_z_embed.py",
    "scripts/run_description_agent.py",
):
    if not (ROOT / rel).is_file():
        raise FileNotFoundError(ROOT / rel)

(ROOT / "data" / "outputs").mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)
print("VISUALHISTORY_ROOT =", vh)
print("faiss OK:", getattr(faiss, "__version__", "?"))
print("VisualHistory OK")
print("required artifacts OK")


In [ ]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()
assert os.environ["OPENAI_API_KEY"], "OPENAI_API_KEY is empty in the config cell"
print("OPENAI_API_KEY loaded")


## 2 — Description Agent (notebook 02)


In [ ]:
%pip install -q "faiss-cpu>=1.7"

import importlib
import visual_history_agent.describe as _desc
import visual_history_agent.encode as _enc
import visual_history_agent.stream_subprocess as _stream

importlib.reload(_stream)
importlib.reload(_enc)
importlib.reload(_desc)

from visual_history_agent.paths import artifact_paths
from visual_history_agent.describe import describe_image

paths = artifact_paths(ROOT)
IMAGE = paths["examples"] / "The_Blue_Room.jpeg"
print(f"Describe starting: {IMAGE.name} on {DEVICE}", flush=True)

describe_image(
    IMAGE,
    coordinator_backend="openai",
    coordinator_model="gpt-5.1",
    device=DEVICE,
    compile_pdf=False,
    print_description=True,
)


## 3 — Comparison (notebook 03)


In [ ]:
import importlib
import visual_history_agent.compare as _cmp
import visual_history_agent.describe as _desc
import visual_history_agent.encode as _enc
import visual_history_agent.stream_subprocess as _stream

importlib.reload(_stream)
importlib.reload(_enc)
importlib.reload(_desc)
importlib.reload(_cmp)

from visual_history_agent.compare import compare_images

IMAGE_A = paths["examples"] / "Norton_Jerome.jpg"
IMAGE_B = paths["examples"] / "Princeton_Jerome.jpg"
print(f"Compare starting: {IMAGE_A.name} vs {IMAGE_B.name} on {DEVICE}", flush=True)

compare_images(
    IMAGE_A,
    IMAGE_B,
    coordinator_backend="openai",
    coordinator_model="gpt-5.1",
    device=DEVICE,
    compile_pdf=False,
)


## 4 — Spatial atom browser (notebook 04)

Builds NMF + local Dai maps (no shipped spatial caches).


In [ ]:
from visual_history_agent.spatial_browser import (
    prepare_spatial_browser,
    display_spatial_atom_browser,
)

bundle = prepare_spatial_browser(
    IMAGE_A,
    IMAGE_B,
    force_recompute=False,
    device=DEVICE,
)
display_spatial_atom_browser(bundle)


In [ ]:
single_bundle = prepare_spatial_browser(
    IMAGE,
    force_recompute=False,
    device=DEVICE,
)
display_spatial_atom_browser(single_bundle)
